In [0]:
%pip install databricks-vectorsearch

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-fd7fa572-a856-459e-8999-19f50ba5464b
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-fd7fa572-a856-459e-8999-19f50ba5464b
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/env

In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks Notebook: 02_vector_search_setup
import time
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

CATALOG = "rag"
SCHEMA = "wb_operations"
ENDPOINT_NAME = "wb_ops_vector_endpoint"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.gold_wb_pad_chunks_index"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.gold_wb_pad_chunks"

def setup_vector_search():
    # 1. Provision Endpoint
    existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
    if ENDPOINT_NAME not in existing_endpoints:
        print(f"Creating endpoint '{ENDPOINT_NAME}'... (may take 10-15 minutes)")
        vsc.create_endpoint_and_wait(name=ENDPOINT_NAME, endpoint_type="STANDARD")
    else:
        print(f"Vector search endpoint '{ENDPOINT_NAME}' is ready.")

    # 2. Check or Create Delta Sync Managed Index
    try:
        index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        print("Vector index already exists.")
        
        # Check if index is already operational before syncing
        status_info = index.describe()
        detailed_state = status_info.get("status", {}).get("detailed_state", "")
        
        if detailed_state in ["ONLINE", "READY", "ONLINE_UPDATING_PIPELINE_RESOURCES"]:
            print(f"Index is already operational (state: {detailed_state}). Skipping sync.")
        else:
            print(f"Index state: {detailed_state}. Triggering sync...")
            index.sync()
    except Exception:
        print(f"Creating Managed Delta Sync Index '{INDEX_NAME}'...")
        vsc.create_delta_sync_index(
            endpoint_name=ENDPOINT_NAME,
            index_name=INDEX_NAME,
            source_table_name=SOURCE_TABLE,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",
            embedding_source_column="chunk_text",
            embedding_model_endpoint_name="databricks-bge-large-en"
        )
        print("Index creation job submitted.")

    # 3. Wait for Provisioning
    while True:
        status_info = vsc.get_index(ENDPOINT_NAME, INDEX_NAME).describe()
        index_status = status_info.get("status", {}).get("detailed_state", "")
        print(f"Current Index State: {index_status}")
        if index_status in ["ONLINE", "READY", "ONLINE_UPDATING_PIPELINE_RESOURCES"]:
            print("Vector Index is fully operational!")
            break
        elif "FAILED" in index_status.upper():
            raise RuntimeError(f"Index provisioning failed: {status_info.get('status', {}).get('message')}")
        time.sleep(20)

setup_vector_search()

/home/spark-fd7fa572-a856-459e-8999-19/.ipykernel/74/command-8023268742383677-1754754331:3: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Vector search endpoint 'wb_ops_vector_endpoint' is ready.
Vector index already exists.
Index state: ONLINE_PIPELINE_FAILED. Triggering sync...
Current Index State: ONLINE_UPDATING_PIPELINE_RESOURCES
Vector Index is fully operational!
